# R17: Baseline (CE + 0.5*Dice + CopyPaste)
使用重构后的 train.py + train/val/test 三分

In [ ]:
# Cell 1: 安装依赖
!pip install rasterio segmentation-models-pytorch -q

In [ ]:
# Cell 2: 检查 GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# Cell 3: 复制代码到工作目录 (Kaggle input 只读)
import os, shutil, sys

DATA_ROOT = '/kaggle/input/datasets/changyasong/datasetv6/datasetv6'
CODE_ROOT = '/kaggle/input/datasets/changyasong/datasetv6/STDL-Net'

# 复制代码
CODE_DST = '/kaggle/working/code'
os.makedirs(CODE_DST, exist_ok=True)

for root, dirs, files in os.walk(CODE_ROOT):
    rel = os.path.relpath(root, CODE_ROOT)
    dst_dir = os.path.join(CODE_DST, rel) if rel != '.' else CODE_DST
    os.makedirs(dst_dir, exist_ok=True)
    for f in files:
        if f.endswith('.py') or f.endswith('.yaml') or f.endswith('.txt'):
            src = os.path.join(root, f)
            dst = os.path.join(dst_dir, f)
            shutil.copy2(src, dst)
            print(f'  copied: {os.path.relpath(dst, CODE_DST)}')

# 确保 configs/ 目录存在，yaml 放入其中
configs_dir = os.path.join(CODE_DST, 'configs')
os.makedirs(configs_dir, exist_ok=True)
for f in os.listdir(CODE_DST):
    if f.endswith('.yaml'):
        src = os.path.join(CODE_DST, f)
        dst = os.path.join(configs_dir, f)
        shutil.move(src, dst)
        print(f'  moved: {f} -> configs/{f}')

print(f'\nCode copied to: {CODE_DST}')
for name in ['train.py', 'configs/R17.yaml']:
    p = os.path.join(CODE_DST, name)
    print(f'  {name}: {"OK" if os.path.isfile(p) else "MISSING"}')

In [ ]:
# Cell 4: 设置预训练权重路径
import os
os.environ['SWIN_PRETRAIN_DIR'] = '/kaggle/input/datasets/changyasong/datasetv6/datasetv6/pretrain'
print(f'SWIN_PRETRAIN_DIR = {os.environ["SWIN_PRETRAIN_DIR"]}')
print(f'Files: {os.listdir(os.environ["SWIN_PRETRAIN_DIR"])}')

In [ ]:
# Cell 5: 运行训练
!cd /kaggle/working/code && python train.py --config configs/R17.yaml

In [ ]:
# Cell 6: 打包结果
import zipfile, os

RESULT_DIR = '/kaggle/working/result'
zip_path = '/kaggle/working/result.zip'
if os.path.isdir(RESULT_DIR):
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(RESULT_DIR):
            for f in files:
                fpath = os.path.join(root, f)
                arcname = os.path.relpath(fpath, '/kaggle/working')
                zf.write(fpath, arcname)
    print(f'Packed: {zip_path}')
    !ls -lh /kaggle/working/result.zip
else:
    print('No result directory found, skipping pack.')

print('Done. Kaggle will auto-release GPU after notebook finishes.')